In [132]:
%pip install transformers
%pip install torch

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [133]:
import pandas as pd
import numpy as np
np.random.seed(123)

In [134]:
base_data = pd.read_csv('project_data/full_dataset.csv')
base_data = base_data.loc[base_data.created_at < "2020-05-01"]
base_data = base_data.loc[~base_data.comorbid.fillna(True)]
base_data = base_data.loc[base_data.female_prop.abs() > 0.8]
base_data = base_data.loc[base_data.group_anxiety | base_data.group_depression]
base_data = base_data.loc[~(base_data.group_anxiety & base_data.group_depression)]

/var/folders/7d/2d_kq54s00v0272gkndgzxd80000gn/T/ipykernel_4069/1376200099.py:1: DtypeWarning: Columns (27,28,29,33,34,35) have mixed types. Specify dtype option on import or set low_memory=False.
  base_data = pd.read_csv('project_data/full_dataset.csv')
/var/folders/7d/2d_kq54s00v0272gkndgzxd80000gn/T/ipykernel_4069/1376200099.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  base_data = base_data.loc[~base_data.comorbid.fillna(True)]


In [135]:
base_data

,tweet_id,user_id,created_at,ANEW_Valence,ANEW_Dominance,ANEW_Arousal,Happiness,NRC_anger,NRC_anticipation,NRC_disgust,...,comorbid,cohort,inferred_timezone,timezone_loss,day_of_week,female_prop,gender,VADER_NEG,VADER_NEU,VADER_POS
4082,tD2620597,uANX341,2020-01-10 18:54:32+00:00,6.041,6.004,4.0630,5.631200,1.0,1.0,0.0,...,False,anxiety,-4,0.016495,4,0.9992,female,0.355,0.507,0.138
4083,tD2620598,uANX341,2020-01-10 18:45:31+00:00,6.545,5.640,3.6450,5.559048,0.0,0.0,0.0,...,False,anxiety,-4,0.016495,4,0.9992,female,0.000,1.000,0.000
4084,tD2620599,uANX341,2020-01-03 11:21:27+00:00,5.015,4.915,4.0850,5.151429,0.0,1.0,0.0,...,False,anxiety,-4,0.016495,4,0.9992,female,0.156,0.655,0.189
4085,tD2620600,uANX341,2019-12-25 01:20:32+00:00,6.635,6.455,4.0450,6.056000,0.0,0.0,0.0,...,False,anxiety,-4,0.016495,2,0.9992,female,0.270,0.317,0.413
4086,tD2620601,uANX341,2019-12-24 12:37:31+00:00,6.855,6.355,4.6725,6.169091,0.0,0.0,0.0,...,False,anxiety,-4,0.016495,1,0.9992,female,0.049,0.828,0.123
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2102224,tD2614323,uDEP433,2020-04-18 10:17:50+00:00,6.032,5.670,3.6280,5.132308,0.0,1.0,0.0,...,False,depression,-2,0.029240,5,-0.9978,male,0.000,0.918,0.082
2102225,tD2614324,uDEP433,2020-04-17 22:47:27+00:00,5.865,5.455,3.4300,5.567273,0.0,0.0,0.0,...,False,depression,-2,0.029240,4,-0.9978,male,0.000,0.870,0.130
2102226,tD2614325,uDEP433,2020-04-17 22:23:37+00:00,7.090,5.935,3.8550,6.446667,0.0,0.0,0.0,...,False,depression,-2,0.029240,4,-0.9978,male,0.000,0.872,0.128
2102227,tD2614326,uDEP433,2020-04-17 15:36:36+00:00,6.380,6.560,3.5700,5.460000,0.0,0.0,0.0,...,False,depression,-2,0.029240,4,-0.9978,male,0.000,0.704,0.296


In [136]:
depression_patients = base_data.loc[base_data.group_depression.fillna(False)].user_id.unique()
anxiety_patients = base_data.loc[base_data.group_anxiety.fillna(False)].user_id.unique()

/var/folders/7d/2d_kq54s00v0272gkndgzxd80000gn/T/ipykernel_4069/3913742314.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  depression_patients = base_data.loc[base_data.group_depression.fillna(False)].user_id.unique()
/var/folders/7d/2d_kq54s00v0272gkndgzxd80000gn/T/ipykernel_4069/3913742314.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  anxiety_patients = base_data.loc[base_data.group_anxiety.fillna(False)].user_id.unique()


In [137]:
anx_choice = np.random.choice(anxiety_patients, 250, replace=False)
dep_choice = np.random.choice(depression_patients, 250, replace=False)

In [138]:
tweets_list : list[pd.DataFrame] = []
import os
data_files : list = os.listdir('data')
diagnoses : list[str] = list(filter(lambda file: '.' not in file, data_files))

for diagnosis in diagnoses:
    tweets_list.append(
        pd.read_csv(
            f'data_with_text/{diagnosis}/texts.tsv',
            sep = '\t',
            header=0,
            index_col=0
        )
    )

full_tweets : pd.DataFrame = pd.concat(tweets_list).reset_index()

full_tweets

,tweet_id,user_id,full_text
0,tD2616022,uANX100,my niece just said “oh look what I’m gonna do”...
1,tD2616023,uANX100,Some old bald guy yanked my ponytail at the ba...
2,tD2616024,uANX100,Somebody thought it would be a good idea to le...
3,tD2616025,uANX100,"Due to personal reasons, everybody pls leave m..."
4,tD2616026,uANX100,I really get home from work and take a shower ...
...,...,...,...
4079750,tD0139188,uPDD001,omg remember when jimin https://t.co/lDBUnTH2H6
4079751,tD0139189,uPDD001,dimple https://t.co/RDzjR80JiV
4079752,tD0139190,uPDD001,this didn’t age well
4079753,tD0139191,uPDD001,jimi pointing @ sleeping jimi https://t.co/Op4...


In [139]:
anx_text_choice = full_tweets.loc[full_tweets.user_id.isin(anx_choice)]
dep_text_choice = full_tweets.loc[full_tweets.user_id.isin(dep_choice)]

print(f"{len(anx_text_choice) = }, {len(dep_text_choice) = }")

len(anx_text_choice) = 357099, len(dep_text_choice) = 318493


In [140]:
text_data = pd.concat(
    [
        anx_text_choice,
        dep_text_choice
    ]
)

In [141]:
import torch
import pandas as pd
import time
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.nn.functional import softmax
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# Load tokenizer and model
model_str = 'cardiffnlp/twitter-roberta-base-sentiment-latest'
tokenizer = AutoTokenizer.from_pretrained(model_str)
model = AutoModelForSequenceClassification.from_pretrained(model_str)

# Set device
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")
model.to(device)
model.eval()

# Input dataset
class TweetDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return row['tweet_id'], row['full_text']

# Collate function with dynamic padding
def collate_fn(batch):
    tweet_ids, texts = zip(*batch)
    encoded = tokenizer(
        list(texts),
        return_tensors='pt',
        padding=True,          # dynamically pad to max in batch
        truncation=True,
        max_length=512         # ensure nothing goes beyond model limit
    )
    return tweet_ids, encoded

# Load your data
sample_data = text_data[['tweet_id', 'full_text']].dropna().reset_index(drop=True)
dataset = TweetDataset(sample_data)
dataloader = DataLoader(dataset, batch_size=32, collate_fn=collate_fn)

# Inference
results = []
start = time.time()

for tweet_ids, batch in tqdm(dataloader, desc="Running inference", total=len(dataloader)):
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        logits = model(**batch).logits
        probs = softmax(logits, dim=-1).cpu()

    for tweet_id, prob in zip(tweet_ids, probs):
        results.append({
            'tweet_id': tweet_id,
            'neg_prob': prob[0].item(),
            'neu_prob': prob[1].item(),
            'pos_prob': prob[2].item()
        })

print(f"\nProcessed {len(results)} tweets in {time.time() - start:.2f} seconds.")
results_df = pd.DataFrame(results)


Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Using device: mps


Running inference: 100%|██████████| 21113/21113 [1:23:12<00:00,  4.23it/s]  



Processed 675592 tweets in 4992.44 seconds.


In [142]:
results_df

,tweet_id,neg_prob,neu_prob,pos_prob
0,tD2620586,0.938503,0.056360,0.005137
1,tD2620587,0.439552,0.401197,0.159251
2,tD2620588,0.003731,0.014676,0.981594
3,tD2620589,0.366901,0.604518,0.028581
4,tD2620590,0.006213,0.267819,0.725967
...,...,...,...,...
675587,tD2605669,0.029113,0.883204,0.087684
675588,tD2605670,0.018828,0.531654,0.449518
675589,tD2605671,0.010002,0.227946,0.762051
675590,tD2605672,0.007349,0.815530,0.177121


In [151]:
data = pd.merge(
    base_data,
    results_df,
    how = 'inner',
    on = 'tweet_id',
).rename(columns={'neg_prob':'ROBERTA_NA'})

data

,tweet_id,user_id,created_at,ANEW_Valence,ANEW_Dominance,ANEW_Arousal,Happiness,NRC_anger,NRC_anticipation,NRC_disgust,...,timezone_loss,day_of_week,female_prop,gender,VADER_NEG,VADER_NEU,VADER_POS,ROBERTA_NA,neu_prob,pos_prob
0,tD2620597,uANX341,2020-01-10 18:54:32+00:00,6.041,6.004000,4.063000,5.631200,1.0,1.0,0.0,...,0.016495,4,0.9992,female,0.355,0.507,0.138,0.643774,0.279017,0.077209
1,tD2620598,uANX341,2020-01-10 18:45:31+00:00,6.545,5.640000,3.645000,5.559048,0.0,0.0,0.0,...,0.016495,4,0.9992,female,0.000,1.000,0.000,0.260429,0.354843,0.384729
2,tD2620599,uANX341,2020-01-03 11:21:27+00:00,5.015,4.915000,4.085000,5.151429,0.0,1.0,0.0,...,0.016495,4,0.9992,female,0.156,0.655,0.189,0.867747,0.119338,0.012915
3,tD2620600,uANX341,2019-12-25 01:20:32+00:00,6.635,6.455000,4.045000,6.056000,0.0,0.0,0.0,...,0.016495,2,0.9992,female,0.270,0.317,0.413,0.002843,0.010982,0.986175
4,tD2620601,uANX341,2019-12-24 12:37:31+00:00,6.855,6.355000,4.672500,6.169091,0.0,0.0,0.0,...,0.016495,1,0.9992,female,0.049,0.828,0.123,0.003068,0.007406,0.989526
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
493910,tD2605669,uDEP149,2019-10-20 14:53:04+00:00,NaN,NaN,NaN,NaN,0.0,0.0,0.0,...,0.014907,6,0.9974,female,0.292,0.708,0.000,0.029113,0.883204,0.087684
493911,tD2605670,uDEP149,2019-10-20 10:49:58+00:00,6.050,6.190000,5.150000,6.480000,0.0,1.0,0.0,...,0.014907,6,0.9974,female,0.218,0.565,0.218,0.018828,0.531654,0.449518
493912,tD2605671,uDEP149,2019-10-20 00:43:09+00:00,NaN,NaN,NaN,5.372000,0.0,1.0,0.0,...,0.014907,6,0.9974,female,0.000,1.000,0.000,0.010002,0.227946,0.762051
493913,tD2605672,uDEP149,2019-10-19 12:25:20+00:00,6.135,5.750000,3.050000,5.992000,0.0,0.0,0.0,...,0.014907,5,0.9974,female,0.000,1.000,0.000,0.007349,0.815530,0.177121


# Full data

In [152]:
data.drop_duplicates(subset=['user_id']).groupby('group_anxiety').count()

,tweet_id,user_id,created_at,ANEW_Valence,ANEW_Dominance,ANEW_Arousal,Happiness,NRC_anger,NRC_anticipation,NRC_disgust,...,timezone_loss,day_of_week,female_prop,gender,VADER_NEG,VADER_NEU,VADER_POS,ROBERTA_NA,neu_prob,pos_prob
group_anxiety,,,,,,,,,,,,,,,,,,,,,
False,250,250,250,207,207,207,228,250,250,250,...,250,250,250,250,250,250,250,250,250,250
True,230,230,230,191,191,191,215,230,230,230,...,230,230,230,230,230,230,230,230,230,230


In [153]:
tweet_counts = data.groupby('user_id').agg({
    'cohort' : 'first',
    'tweet_id' : 'count'
}).reset_index().rename({
    'tweet_id' : 'tweet_count'
}, axis=1)

anxiety_users = tweet_counts.loc[tweet_counts.cohort == 'anxiety'].sample(230, random_state=123)
depression_users = tweet_counts.loc[tweet_counts.cohort == 'depression']

depression_users_raw = []

for idx, row in anxiety_users.iterrows():
    depression_users['diff'] = (depression_users['tweet_count'] - row['tweet_count']).abs()
    best_match = depression_users.loc[depression_users['diff'] == depression_users['diff'].min()]
    depression_users_raw.append(best_match.iloc[0].to_dict())
    depression_users = depression_users.loc[depression_users.user_id != best_match.iloc[0].user_id]

new_data = pd.concat([anxiety_users, pd.DataFrame(depression_users_raw)])

data = data.loc[data.user_id.isin(new_data.user_id)]

data

/var/folders/7d/2d_kq54s00v0272gkndgzxd80000gn/T/ipykernel_4069/1157249342.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  depression_users['diff'] = (depression_users['tweet_count'] - row['tweet_count']).abs()


,tweet_id,user_id,created_at,ANEW_Valence,ANEW_Dominance,ANEW_Arousal,Happiness,NRC_anger,NRC_anticipation,NRC_disgust,...,timezone_loss,day_of_week,female_prop,gender,VADER_NEG,VADER_NEU,VADER_POS,ROBERTA_NA,neu_prob,pos_prob
0,tD2620597,uANX341,2020-01-10 18:54:32+00:00,6.041,6.004000,4.063000,5.631200,1.0,1.0,0.0,...,0.016495,4,0.9992,female,0.355,0.507,0.138,0.643774,0.279017,0.077209
1,tD2620598,uANX341,2020-01-10 18:45:31+00:00,6.545,5.640000,3.645000,5.559048,0.0,0.0,0.0,...,0.016495,4,0.9992,female,0.000,1.000,0.000,0.260429,0.354843,0.384729
2,tD2620599,uANX341,2020-01-03 11:21:27+00:00,5.015,4.915000,4.085000,5.151429,0.0,1.0,0.0,...,0.016495,4,0.9992,female,0.156,0.655,0.189,0.867747,0.119338,0.012915
3,tD2620600,uANX341,2019-12-25 01:20:32+00:00,6.635,6.455000,4.045000,6.056000,0.0,0.0,0.0,...,0.016495,2,0.9992,female,0.270,0.317,0.413,0.002843,0.010982,0.986175
4,tD2620601,uANX341,2019-12-24 12:37:31+00:00,6.855,6.355000,4.672500,6.169091,0.0,0.0,0.0,...,0.016495,1,0.9992,female,0.049,0.828,0.123,0.003068,0.007406,0.989526
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
493910,tD2605669,uDEP149,2019-10-20 14:53:04+00:00,NaN,NaN,NaN,NaN,0.0,0.0,0.0,...,0.014907,6,0.9974,female,0.292,0.708,0.000,0.029113,0.883204,0.087684
493911,tD2605670,uDEP149,2019-10-20 10:49:58+00:00,6.050,6.190000,5.150000,6.480000,0.0,1.0,0.0,...,0.014907,6,0.9974,female,0.218,0.565,0.218,0.018828,0.531654,0.449518
493912,tD2605671,uDEP149,2019-10-20 00:43:09+00:00,NaN,NaN,NaN,5.372000,0.0,1.0,0.0,...,0.014907,6,0.9974,female,0.000,1.000,0.000,0.010002,0.227946,0.762051
493913,tD2605672,uDEP149,2019-10-19 12:25:20+00:00,6.135,5.750000,3.050000,5.992000,0.0,0.0,0.0,...,0.014907,5,0.9974,female,0.000,1.000,0.000,0.007349,0.815530,0.177121


In [154]:
data.to_csv('project_data/full_dataset_roberta.csv', index = False)

# 6-hour-windowed data

In [155]:
def get_windows(subdf):
    subdf.index = pd.to_datetime(subdf['created_at'])
    subdf = subdf.sort_index()
    subdf['gender_female'] = subdf['gender'].map(lambda x: 1 if x == 'female' else 0)
    #print(subdf.index)
    return subdf.rolling(
        window = '6h',
        min_periods = 2,
        center=True
    ).agg({
        'ROBERTA_NA': 'mean',
        'group_anxiety': 'min',
        'group_depression': 'min',
        'comorbid':'min',
        'gender_female' : 'min',
        '19-29' : 'mean',
        '30-39' : 'mean',
        '<=18' : 'mean',
        '>=40' : 'mean',
        'day_of_week' : 'median'
    }).dropna()

data_aggregated = data.groupby('user_id').apply(get_windows).reset_index()

data_aggregated['gender'] = data_aggregated['gender_female'].map(lambda x: 'female' if x == 1 else 'male')

data_aggregated.to_csv('project_data/windowed_mean_data_roberta.csv')

/var/folders/7d/2d_kq54s00v0272gkndgzxd80000gn/T/ipykernel_4069/195955282.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data_aggregated = data.groupby('user_id').apply(get_windows).reset_index()


# Daily data

In [156]:
data['day'] = data['created_at'].map(lambda x: str(x).split(' ')[0])
data['gender_female'] = data['gender'].map(lambda x: 1 if x == 'female' else 0)

data_aggregated = data.groupby(['user_id', 'day']).agg({
        'ROBERTA_NA': 'mean',
        'group_anxiety': 'min',
        'group_depression': 'min',
        'comorbid':'min',
        'gender_female' : 'min',
        '19-29' : 'mean',
        '30-39' : 'mean',
        '<=18' : 'mean',
        '>=40' : 'mean',
        'day_of_week' : 'median'
    }).dropna().reset_index()

data_aggregated['gender'] = data_aggregated['gender_female'].map(lambda x: 'female' if x == 1 else 'male')

data_aggregated.to_csv('project_data/daily_mean_data_roberta.csv')

/var/folders/7d/2d_kq54s00v0272gkndgzxd80000gn/T/ipykernel_4069/539719974.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['day'] = data['created_at'].map(lambda x: str(x).split(' ')[0])
/var/folders/7d/2d_kq54s00v0272gkndgzxd80000gn/T/ipykernel_4069/539719974.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['gender_female'] = data['gender'].map(lambda x: 1 if x == 'female' else 0)


# Save raw scores

In [157]:
results_df.to_csv(
    'project_data/roberta_scores_cardiff.csv'
)

In [169]:
data.groupby('user_id').agg({
    'tweet_id' : 'count',
    'group_anxiety' : 'min',
    "gender_female" : "first",
    'group_depression' : 'min','19-29' : 'mean',
        '30-39' : 'mean',
        '<=18' : 'mean',
        '>=40' : 'mean',}).reset_index().groupby('group_anxiety').agg({
        'user_id' : 'nunique',
        'tweet_id' : ['mean', 'std'],
        'gender_female':'sum',
        '19-29' : 'mean',
        '30-39' : 'mean',
        '<=18' : 'mean',
        '>=40' : 'mean',
    })

user_id     tweet_id             gender_female     19-29  \
              nunique         mean         std           sum      mean   
group_anxiety                                                            
False             230  1030.182609  726.661855           146  0.410923   
True              230  1085.856522  757.097977           163  0.423560   

                  30-39      <=18      >=40  
                   mean      mean      mean  
group_anxiety                                
False          0.174297  0.284587  0.130193  
True           0.233749  0.246627  0.096064